# [1.2] Intro to Mechanistic Interpretability: nnsight & Induction Circuits (Exercises)

This notebook reimplements the [ARENA Chapter 1.2 exercises](https://arena3-chapter1-transformer-interp.streamlit.app/%5B1.2%5D_Intro_to_Mech_Interp) using **nnsight** and **NDIF** for remote execution, instead of TransformerLens.

The model, weights, and exercises are identical - only the intervention/hooking API changes.

## Introduction

These exercises are designed to get you introduced to the core concepts of mechanistic interpretability, using **nnsight** (with NDIF remote execution) as the primary tool.

The running theme of the exercises is **induction circuits**. Induction circuits are a particular type of circuit in a transformer, which can perform basic in-context learning. You should read the [corresponding section of Neel's glossary](https://dynalist.io/d/n2ZWtnoYHrU1s4vnFSAQ519J#z=_Jzi6YHRHKP1JziwdE02qdYZ) before continuing. This [LessWrong post](https://www.lesswrong.com/posts/TvrfY4c9eaGLeyDkE/induction-heads-illustrated) might also help; it contains some diagrams which walk through the induction mechanism step by step.

Each exercise will have a difficulty and importance rating out of 5, as well as an estimated maximum time you should spend on these exercises. Please do skip exercises / look at solutions if you don't feel like they're important enough to be worth doing, and you'd rather get to the good stuff!

### Key Difference: nnsight vs TransformerLens

| TransformerLens | nnsight equivalent |
|---|---|
| `logits, cache = model.run_with_cache(tokens)` | `with model.trace(tokens): val = model.module.output.save()` |
| `cache["pattern", layer]` | `model.blocks[layer].attn.hook_pattern.output.save()` |
| `cache["result", layer]` | `model.blocks[layer].attn.hook_result.output.save()` |
| `model.run_with_hooks(tokens, fwd_hooks=[(name, fn)])` | Modify outputs directly inside `with model.trace():` |
| `z[:, :, head, :] = 0.0` (in hook) | Same code, inside `with model.trace():` |

## Content & Learning Objectives

### Section 1: Introduction to nnsight

This section introduces nnsight's core API using GPT-2 Small: loading models, running traces, and saving activations.

> ##### Learning Objectives
>
> - Load and run models with nnsight's `LanguageModel` and `NNsight` wrappers
> - Understand the `model.trace()` context for deferred execution
> - Use `.output.save()` to capture activations
> - Use `circuitsvis` to visualise attention heads

### Section 2: Finding induction heads

Here, you'll learn about induction heads, how they work and why they are important. You'll inspect activation patterns and identify induction heads.

> ##### Learning Objectives
>
> - Understand what induction heads are, and the algorithm they are implementing
> - Inspect activation patterns to identify basic attention head patterns, and write your own functions to detect attention heads
> - Identify induction heads by looking at the attention patterns produced from a repeating random sequence

### Section 3: nnsight Interventions

Next, you'll learn how nnsight's trace context replaces TransformerLens hooks. You'll build attribution tools and perform ablation studies.

> ##### Learning Objectives
>
> - Use nnsight's trace context to access and modify activations
> - Build tools to perform attribution, i.e. detecting which components are responsible for performance on a given task
> - Understand how to perform basic interventions like **ablation** using nnsight

### Section 4: Reverse-engineering induction circuits

Lastly, you'll reverse-engineer the induction circuit by looking directly at the transformer's weights. You'll examine QK and OV circuits, composition scores, and targeted ablations.

> ##### Learning Objectives
>
> - Use the FactoredMatrix class to inspect the QK and OV circuits within an induction circuit
> - Understand composition between attention heads, and how to measure it
> - Perform targeted ablations to verify the circuit

## Setup

In [1]:
# Cell 1: Install dependencies
%pip install nnsight einops circuitsvis plotly jaxtyping huggingface_hub transformers eindex-callum torch nbformat

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 23.2.1 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
# Cell 2: Imports and NDIF Configuration
import torch
import torch.nn as nn
import torch.nn.functional as F
import einops
import numpy as np
import functools
from dataclasses import dataclass
from typing import Callable, Optional
from tqdm import tqdm
import plotly.express as px
import plotly.graph_objects as go
import plotly.io as pio
from IPython.display import display
import circuitsvis as cv
from nnsight import NNsight, LanguageModel, CONFIG
from transformers import AutoTokenizer
from huggingface_hub import hf_hub_download
from eindex import eindex
from jaxtyping import Float, Int
from torch import Tensor

# Configure plotly renderer to avoid nbformat detection bug
pio.renderers.default = "plotly_mimetype+notebook"

# Configure NDIF API key for remote execution
# Sign up at https://login.ndif.us/ and replace with your key
CONFIG.set_default_api_key("YOUR_NDIF_API_KEY")

torch.set_grad_enabled(False)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

c:\Users\anish\AppData\Local\Programs\Python\Python312\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Using device: cpu


## nnsight Primer

**nnsight** is a library for intercepting and modifying neural network activations. It uses a **deferred execution** model via the `model.trace()` context manager.

**Key patterns:**
- `with model.trace(input):` - opens a trace context where operations are recorded
- `model.module.output.save()` - saves the output of a module for later use
- `model.module.output[:, :, head, :] = 0.0` - modifies activations in-place (interventions)
- Variables must be initialized **before** the trace context (they don't persist if created inside)
- Modules must be accessed in **forward-pass order** inside the trace

**NDIF** (National Deep Inference Facility) provides remote GPU execution. When configured with an API key, nnsight automatically runs models on NDIF's remote GPUs - no local GPU required.

## Plotting Utilities

In [3]:
# Cell 3: Plotting utilities
def to_numpy(tensor):
    if isinstance(tensor, np.ndarray):
        return tensor
    return tensor.detach().cpu().float().numpy()

def imshow(tensor, title="", labels=None, text_auto=False, width=None, height=None,
           x=None, y=None, return_fig=False, **kwargs):
    fig = px.imshow(
        to_numpy(tensor), title=title, labels=labels or {},
        color_continuous_midpoint=0, text_auto=text_auto,
        width=width, height=height, x=x, y=y, **kwargs
    )
    if return_fig:
        return fig
    fig.show()

def line(y_values, title="", x=None, labels=None, width=None, height=None, **kwargs):
    fig = px.line(y=to_numpy(y_values), x=x, title=title, labels=labels or {},
                  width=width, height=height, **kwargs)
    fig.show()

def hist(values, title="", nbins=50, width=None, labels=None, **kwargs):
    fig = px.histogram(x=to_numpy(values) if isinstance(values, (torch.Tensor, np.ndarray)) else values,
                       title=title, nbins=nbins, width=width, labels=labels or {}, **kwargs)
    fig.show()

## Model Architecture

We reimplement the `attn_only_2L_half` model as a plain `nn.Module`, matching the TransformerLens architecture exactly:
- 2 layers, 12 heads, d_model=768, d_head=64
- Attention-only (no MLPs), no LayerNorm
- **Shortformer** positional embeddings: pos_embed is added to Q and K inputs at each layer, but NOT to V or the residual stream

This means that **the residual stream can't directly encode positional information**. This turns out to make it much easier for induction heads to form.

`HookPoint` modules (identity pass-throughs) are placed at key intermediate values so nnsight can intercept them via `.output`.

In [4]:
# Cell 4: Config, HookPoint, Embed, PosEmbed, Unembed
@dataclass
class ModelConfig:
    d_model: int = 768
    d_head: int = 64
    n_heads: int = 12
    n_layers: int = 2
    n_ctx: int = 2048
    d_vocab: int = 50278
    tokenizer_name: str = "EleutherAI/gpt-neox-20b"


class HookPoint(nn.Module):
    """Identity module. Exists so nnsight can intercept values at this point via .output"""
    def forward(self, x):
        return x


class Embed(nn.Module):
    def __init__(self, cfg: ModelConfig):
        super().__init__()
        self.W_E = nn.Parameter(torch.empty(cfg.d_vocab, cfg.d_model))

    def forward(self, tokens):
        return self.W_E[tokens]


class PosEmbed(nn.Module):
    def __init__(self, cfg: ModelConfig):
        super().__init__()
        self.W_pos = nn.Parameter(torch.empty(cfg.n_ctx, cfg.d_model))

    def forward(self, tokens):
        seq_len = tokens.shape[-1]
        return self.W_pos[:seq_len]


class Unembed(nn.Module):
    def __init__(self, cfg: ModelConfig):
        super().__init__()
        self.W_U = nn.Parameter(torch.empty(cfg.d_model, cfg.d_vocab))
        self.b_U = nn.Parameter(torch.zeros(cfg.d_vocab))

    def forward(self, resid):
        return resid @ self.W_U + self.b_U

In [5]:
# Cell 5: Attention module with hook points
class Attention(nn.Module):
    def __init__(self, cfg: ModelConfig):
        super().__init__()
        self.cfg = cfg
        self.W_Q = nn.Parameter(torch.empty(cfg.n_heads, cfg.d_model, cfg.d_head))
        self.W_K = nn.Parameter(torch.empty(cfg.n_heads, cfg.d_model, cfg.d_head))
        self.W_V = nn.Parameter(torch.empty(cfg.n_heads, cfg.d_model, cfg.d_head))
        self.W_O = nn.Parameter(torch.empty(cfg.n_heads, cfg.d_head, cfg.d_model))
        self.b_Q = nn.Parameter(torch.zeros(cfg.n_heads, cfg.d_head))
        self.b_K = nn.Parameter(torch.zeros(cfg.n_heads, cfg.d_head))
        self.b_V = nn.Parameter(torch.zeros(cfg.n_heads, cfg.d_head))
        self.b_O = nn.Parameter(torch.zeros(cfg.d_model))
        self.register_buffer("mask", torch.tril(torch.ones(cfg.n_ctx, cfg.n_ctx, dtype=torch.bool)))
        self.register_buffer("IGNORE", torch.tensor(float("-inf")))
        # Hook points for nnsight interception
        self.hook_q = HookPoint()
        self.hook_k = HookPoint()
        self.hook_v = HookPoint()
        self.hook_pattern = HookPoint()
        self.hook_z = HookPoint()
        self.hook_result = HookPoint()

    def forward(self, resid_pre, shortformer_pos_embed):
        # Shortformer: add pos_embed to Q and K inputs only
        q_input = resid_pre + shortformer_pos_embed
        k_input = resid_pre + shortformer_pos_embed
        v_input = resid_pre  # V does NOT get positional embeddings

        q = einops.einsum(q_input, self.W_Q, "b p m, h m d -> b p h d") + self.b_Q
        k = einops.einsum(k_input, self.W_K, "b p m, h m d -> b p h d") + self.b_K
        v = einops.einsum(v_input, self.W_V, "b p m, h m d -> b p h d") + self.b_V

        q = self.hook_q(q)
        k = self.hook_k(k)
        v = self.hook_v(v)

        attn_scores = einops.einsum(q, k, "b qp h d, b kp h d -> b h qp kp") / (self.cfg.d_head ** 0.5)
        seq_len = resid_pre.shape[1]
        attn_scores = torch.where(self.mask[:seq_len, :seq_len], attn_scores, self.IGNORE)

        pattern = F.softmax(attn_scores, dim=-1)
        pattern = torch.where(torch.isnan(pattern), torch.zeros_like(pattern), pattern)
        pattern = self.hook_pattern(pattern)

        z = einops.einsum(pattern, v, "b h qp kp, b kp h d -> b qp h d")
        z = self.hook_z(z)

        result = einops.einsum(z, self.W_O, "b p h d, h d m -> b p h m")
        result = self.hook_result(result)

        attn_out = result.sum(dim=2) + self.b_O
        return attn_out

In [6]:
# Cell 6: Block and full AttnOnly2L model
class Block(nn.Module):
    def __init__(self, cfg: ModelConfig):
        super().__init__()
        self.attn = Attention(cfg)

    def forward(self, resid_pre, shortformer_pos_embed):
        attn_out = self.attn(resid_pre, shortformer_pos_embed)
        return resid_pre + attn_out


class AttnOnly2L(nn.Module):
    def __init__(self, cfg: ModelConfig):
        super().__init__()
        self.cfg = cfg
        self.embed = Embed(cfg)
        self.pos_embed = PosEmbed(cfg)
        self.blocks = nn.ModuleList([Block(cfg) for _ in range(cfg.n_layers)])
        self.unembed = Unembed(cfg)

    def forward(self, tokens):
        token_embed = self.embed(tokens)
        pos_embed = self.pos_embed(tokens)
        resid = token_embed
        for block in self.blocks:
            resid = block(resid, pos_embed)
        return self.unembed(resid)

    @classmethod
    def from_pretrained(cls, device="cpu"):
        cfg = ModelConfig()
        model = cls(cfg)
        weights_path = hf_hub_download(
            repo_id="callummcdougall/attn_only_2L_half",
            filename="attn_only_2L_half.pth",
        )
        pretrained = torch.load(weights_path, map_location=device, weights_only=True)
        missing, unexpected = model.load_state_dict(pretrained, strict=False)
        print(f"Missing keys (expected - hook points have no params): {missing}")
        print(f"Unexpected keys (expected - checkpoint buffers): {unexpected}")
        return model.to(device)

## FactoredMatrix Utility

A lightweight reimplementation of TransformerLens's `FactoredMatrix` for efficient circuit analysis without materializing large $(d_{\text{vocab}} \times d_{\text{vocab}})$ matrices.

In transformer interpretability, we often need to analyse low rank factorized matrices - a matrix $M = AB$, where $M$ is `[large, large]`, but $A$ is `[large, small]` and $B$ is `[small, large]`. The `FactoredMatrix` class implements efficient algorithms for computing the trace, eigenvalues, Frobenius norm, SVD, and products without materializing the full matrix.

In [7]:
# Cell 7: FactoredMatrix utility
class FactoredMatrix:
    def __init__(self, A: Tensor, B: Tensor):
        self.A = A
        self.B = B

    @property
    def AB(self) -> Tensor:
        return self.A @ self.B

    @property
    def T(self) -> "FactoredMatrix":
        return FactoredMatrix(self.B.mT, self.A.mT)

    @property
    def shape(self):
        return (*self.A.shape[:-1], self.B.shape[-1])

    def __matmul__(self, other):
        if isinstance(other, FactoredMatrix):
            return FactoredMatrix(self.A, self.B @ other.A @ other.B)
        elif isinstance(other, Tensor):
            return FactoredMatrix(self.A, self.B @ other)
        return NotImplemented

    def __rmatmul__(self, other):
        if isinstance(other, Tensor):
            return FactoredMatrix(other @ self.A, self.B)
        return NotImplemented

    def __getitem__(self, idx):
        if isinstance(idx, tuple) and len(idx) == 2:
            row_idx, col_idx = idx
            return FactoredMatrix(self.A[..., row_idx, :], self.B[..., :, col_idx])
        elif isinstance(idx, Tensor) and self.A.dim() == 2:
            return FactoredMatrix(self.A[idx], self.B)
        else:
            return FactoredMatrix(self.A[idx], self.B[idx])

    def norm(self) -> Tensor:
        """Efficient Frobenius norm: ||AB||_F without materializing AB."""
        ATA = self.A.mT @ self.A
        BBT = self.B @ self.B.mT
        return (ATA * BBT).sum((-2, -1)).sqrt()

    def get_corner(self, n: int = 10) -> Tensor:
        return self.A[..., :n, :] @ self.B[..., :, :n]

---
# Section 1: Introduction to nnsight

> ##### Learning Objectives
>
> - Load and run models with nnsight's `LanguageModel` and `NNsight` wrappers
> - Understand the `model.trace()` context for deferred execution
> - Use `.output.save()` to capture activations
> - Use `circuitsvis` to visualise attention heads

## Loading and Running Models

In nnsight, we use `LanguageModel` to wrap HuggingFace models (like GPT-2), and `NNsight` to wrap custom `nn.Module` models.

The key difference from TransformerLens:
- **TransformerLens**: `logits = model(text, return_type="logits")`
- **nnsight**: `with model.trace(text): logits = model.lm_head.output.save()`

Models run inside a `trace()` context, where operations are recorded and executed (either locally or on NDIF's remote GPUs).

In [8]:
# Cell 8: Load GPT-2 with nnsight
gpt2 = LanguageModel("openai-community/gpt2", device_map="auto")

In [9]:
# Cell 9: Run GPT-2, compute loss
model_description_text = """## Loading Models

HookedTransformer comes loaded with >40 open source GPT-style models. You can load any of them in with `HookedTransformer.from_pretrained(MODEL_NAME)`. Each model is loaded into the consistent HookedTransformer architecture, designed to be clean, consistent and interpretability-friendly.

For this demo notebook we'll look at GPT-2 Small, an 80M parameter model. To try the model the model out, let's find the loss on this paragraph!"""

# In nnsight, we run the model inside a trace context and save the outputs we need
with gpt2.trace(model_description_text):
    logits = gpt2.lm_head.output.save()

# Compute loss manually
tokens = gpt2.tokenizer(model_description_text, return_tensors="pt")["input_ids"].to(device)
loss = F.cross_entropy(logits[0, :-1], tokens[0, 1:])
print("Model loss:", loss.item())

Model loss: 4.34739351272583


## Tokenization

The tokenizer is stored inside the model, and you can access it using `model.tokenizer` (for `LanguageModel`) or load it separately with `AutoTokenizer` (for custom models wrapped with `NNsight`).

In [10]:
# Cell 10: Tokenization basics
tokenizer_gpt2 = gpt2.tokenizer

print("Tokens for 'gpt2':", tokenizer_gpt2.tokenize("gpt2"))
print("Token IDs:", tokenizer_gpt2.encode("gpt2"))
print("Decoded:", tokenizer_gpt2.decode([50256, 70, 457, 17]))

Tokens for 'gpt2': ['g', 'pt', '2']
Token IDs: [70, 457, 17]
Decoded: <|endoftext|>gpt2


### Exercise - how many tokens does your model guess correctly?

> ```yaml
> Difficulty: \ud83d\udd34\ud83d\udd34\u26aa\u26aa\u26aa
> Importance: \ud83d\udd35\ud83d\udd35\ud83d\udd35\u26aa\u26aa
>
> You should spend up to ~10 minutes on this exercise.
> ```

Consider the `model_description_text` you fed into your model above. How many tokens did your model guess correctly? Which tokens were correct?

<details>
<summary>Hint</summary>

Use `return_type="logits"` to get the model's predictions (in nnsight, you save `gpt2.lm_head.output`), then take argmax across the vocab dimension. Compare the `[:-1]`th elements of predictions with the `[1:]`th elements of the input tokens.

</details>

<details>
<summary>Solution</summary>

```python
with gpt2.trace(model_description_text):
    logits = gpt2.lm_head.output.save()

tokens = gpt2.tokenizer(model_description_text, return_tensors="pt")["input_ids"].squeeze()
prediction = logits[0].argmax(dim=-1).squeeze()[:-1]
true_tokens = tokens[1:].to(device)
is_correct = prediction == true_tokens

print(f"Model accuracy: {is_correct.sum()}/{len(true_tokens)}")
print(f"Correct tokens: {tokenizer_gpt2.batch_decode(prediction[is_correct.cpu()])}")
```
</details>

In [11]:
# Cell 11: Exercise - model accuracy
with gpt2.trace(model_description_text):
    logits = gpt2.lm_head.output.save()

tokens = gpt2.tokenizer(model_description_text, return_tensors="pt")["input_ids"].squeeze()
prediction = logits[0].argmax(dim=-1).squeeze()[:-1]
true_tokens = tokens[1:].to(device)
is_correct = prediction == true_tokens

print(f"Model accuracy: {is_correct.sum()}/{len(true_tokens)}")
print(f"Correct tokens: {tokenizer_gpt2.batch_decode(prediction[is_correct.cpu()])}")

Model accuracy: 35/110
Correct tokens: ['\n', '\n', 'former', ' with', ' models', '.', ' can', ' of', ' them', 'ooked', 'Trans', 'former', '_', 'NAME', '`.', ' model', ' loaded', ' the', 'ed', 'Trans', 'former', ' to', ' be', ' and', '-', '.', '\n', '\n', ' at', 'PT', '-', ',', ',', "'s", ' the']


## Caching Activations

In TransformerLens: `logits, cache = model.run_with_cache(tokens)`

In nnsight: save whatever you need inside the trace context. This is more explicit - you choose exactly which activations to capture, rather than caching everything.

In [12]:
# Cell 12: Cache activations via nnsight trace
gpt2_text = "Natural language processing tasks, such as question answering, machine translation, reading comprehension, and summarization, are typically approached with supervised learning on task-specific datasets."
gpt2_tokens = gpt2.tokenizer(gpt2_text, return_tensors="pt")["input_ids"].to(device)

with gpt2.trace(gpt2_tokens):
    # Save Q, K, V projections from layer 0 attention
    # GPT-2 uses a fused c_attn that outputs [batch, seq, 3*d_model] for Q,K,V
    qkv_l0 = gpt2.transformer.h[0].attn.c_attn.output.save()
    gpt2_logits = gpt2.lm_head.output.save()

print(f"QKV shape: {qkv_l0.shape}")
print(f"Logits shape: {gpt2_logits.shape}")

QKV shape: torch.Size([1, 32, 2304])
Logits shape: torch.Size([1, 32, 50257])


### Exercise - verify attention patterns from Q and K

> ```yaml
> Difficulty: \ud83d\udd34\ud83d\udd34\ud83d\udd34\u26aa\u26aa
> Importance: \ud83d\udd35\ud83d\udd35\ud83d\udd35\u26aa\u26aa
>
> You should spend up to 10-15 minutes on this exercise.
> ```

Verify that you can reconstruct the attention patterns from the saved Q and K vectors. Compute `layer0_pattern_from_q_and_k` by extracting Q and K from the fused `c_attn` output, computing attention scores, applying the causal mask, scaling, and softmaxing.

<details>
<summary>Hint</summary>

GPT-2's fused `c_attn` output has shape `[batch, seq, 3*d_model]`. Split it into Q, K, V along the last dimension (each with size `d_model`). Then reshape to `[n_heads, seq, d_head]` and compute `einsum("hqd,hkd->hqk", q, k) / sqrt(d_head)`.

</details>

<details>
<summary>Solution</summary>

```python
d_model_gpt2 = 768
n_heads_gpt2 = 12
d_head_gpt2 = 64

q, k, v = qkv_l0[0].split(d_model_gpt2, dim=-1)
seq_len_gpt2 = q.shape[0]

q = q.view(seq_len_gpt2, n_heads_gpt2, d_head_gpt2).permute(1, 0, 2)
k = k.view(seq_len_gpt2, n_heads_gpt2, d_head_gpt2).permute(1, 0, 2)

attn_scores = torch.einsum("hqd,hkd->hqk", q, k) / (d_head_gpt2 ** 0.5)
causal_mask = torch.triu(torch.ones(seq_len_gpt2, seq_len_gpt2, dtype=torch.bool, device=device), diagonal=1)
attn_scores.masked_fill_(causal_mask, -1e9)
layer0_pattern = F.softmax(attn_scores, dim=-1)
```
</details>

In [13]:
# Cell 13: Exercise - verify attention patterns from Q and K
d_model_gpt2 = 768
n_heads_gpt2 = 12
d_head_gpt2 = 64

q, k, v = qkv_l0[0].split(d_model_gpt2, dim=-1)
seq_len_gpt2 = q.shape[0]

q = q.view(seq_len_gpt2, n_heads_gpt2, d_head_gpt2).permute(1, 0, 2)
k = k.view(seq_len_gpt2, n_heads_gpt2, d_head_gpt2).permute(1, 0, 2)

attn_scores = torch.einsum("hqd,hkd->hqk", q, k) / (d_head_gpt2 ** 0.5)
causal_mask = torch.triu(torch.ones(seq_len_gpt2, seq_len_gpt2, dtype=torch.bool, device=device), diagonal=1)
attn_scores.masked_fill_(causal_mask, -1e9)
layer0_pattern = F.softmax(attn_scores, dim=-1)

print(f"Attention pattern shape: {layer0_pattern.shape}")
print("Attention pattern computed successfully from Q and K!")

Attention pattern shape: torch.Size([12, 32, 32])
Attention pattern computed successfully from Q and K!


## Visualising Attention Heads

A key insight from the Mathematical Frameworks paper is that we should focus on interpreting the intrinsically interpretable parts of the model - the input tokens, the output logits and the attention patterns. Let's visualize the attention patterns using CircuitsVis.

In [14]:
# Cell 14: Visualize attention heads with circuitsvis
gpt2_str_tokens = gpt2.tokenizer.tokenize(gpt2_text)
full_str_tokens = [gpt2.tokenizer.decode(t) for t in gpt2_tokens[0].tolist()]

print("Layer 0 Head Attention Patterns:")
display(
    cv.attention.attention_heads(
        tokens=full_str_tokens,
        attention=layer0_pattern.cpu(),
        attention_head_names=[f"L0H{i}" for i in range(n_heads_gpt2)],
    )
)

Layer 0 Head Attention Patterns:


---
# Section 2: Finding induction heads

> ##### Learning Objectives
>
> - Understand what induction heads are, and the algorithm they are implementing
> - Inspect activation patterns to identify basic attention head patterns, and write your own functions to detect attention heads
> - Identify induction heads by looking at the attention patterns produced from a repeating random sequence

## Introducing Our Toy Attention-Only Model

Here we introduce a toy 2L attention-only transformer trained specifically for these exercises. Some changes to make them easier to interpret:
- It has only attention blocks (no MLPs, no LayerNorm, no biases).
- The positional embeddings are only added to Q and K inputs (**shortformer** architecture) - the residual stream can't directly encode positional information.
- There are separate embed and unembed matrices (weights are not tied).

We load this model as a plain `nn.Module` and wrap it with `NNsight` for tracing.

In [15]:
# Cell 15: Load 2-layer model, wrap with NNsight
cfg = ModelConfig()
raw_model = AttnOnly2L.from_pretrained(device=device)
model = NNsight(raw_model)

# Load tokenizer separately (NNsight base class doesn't handle tokenization)
tokenizer = AutoTokenizer.from_pretrained(cfg.tokenizer_name)

print(f"Model loaded. Config: {cfg}")

Missing keys (expected - hook points have no params): []
Unexpected keys (expected - checkpoint buffers): []
Model loaded. Config: ModelConfig(d_model=768, d_head=64, n_heads=12, n_layers=2, n_ctx=2048, d_vocab=50278, tokenizer_name='EleutherAI/gpt-neox-20b')


In [16]:
# Cell 16: Run model, save attention patterns
text = "We think that powerful, significantly superhuman machine intelligence is more likely than not to be created this century. If current machine learning techniques were scaled up to this level, we think they would by default produce systems that are deceptive or manipulative, and that no solid plans are known for how to avoid this."

tokens = tokenizer(text, return_tensors="pt")["input_ids"].to(device)

# Initialize variables BEFORE trace (nnsight scope quirk: variables created
# inside the trace context don't persist outside it)
patterns = {}
logits_saved = None

# IMPORTANT: Access modules in forward-pass order inside the trace
with model.trace(tokens):
    for layer in range(cfg.n_layers):
        patterns[layer] = model.blocks[layer].attn.hook_pattern.output.save()
    logits_saved = model.output.save()

print(f"Logits shape: {logits_saved.shape}")
print(f"Attention pattern shape (layer 0): {patterns[0].shape}")

Logits shape: torch.Size([1, 61, 50278])
Attention pattern shape (layer 0): torch.Size([1, 12, 61, 61])


### Exercise - visualise & inspect attention patterns

> ```yaml
> Difficulty: \ud83d\udd34\ud83d\udd34\u26aa\u26aa\u26aa
> Importance: \ud83d\udd35\ud83d\udd35\ud83d\udd35\u26aa\u26aa
>
> You should spend up to ~10 minutes on this exercise.
> ```

Visualise the attention patterns for both layers. What do you notice about the attention heads? You should spot three relatively distinctive basic patterns. What are these patterns, and can you guess why they might be present?

<details>
<summary>Discussion of results</summary>

We notice three basic patterns:
- **prev_token_heads**: attend mainly to the previous token (e.g. head `0.7`)
- **current_token_heads**: attend mainly to the current token (e.g. head `1.6`)
- **first_token_heads**: attend mainly to the first token (e.g. heads `0.3` or `1.4`)

The prev_token_heads and current_token_heads are unsurprising - nearby words have high mutual information. The first_token_heads are used as a "resting position" for heads that only sometimes activate (since attention probabilities must sum to 1).
</details>

In [17]:
# Cell 17: Visualize 2-layer model attention patterns
str_tokens = [tokenizer.decode(t) for t in tokens[0].tolist()]

for layer in range(cfg.n_layers):
    attention_pattern = patterns[layer].squeeze(0)
    display(cv.attention.attention_heads(
        tokens=str_tokens, attention=attention_pattern.cpu(),
        attention_head_names=[f"L{layer}H{i}" for i in range(cfg.n_heads)],
    ))

### Exercise - write your own detectors

> ```yaml
> Difficulty: \ud83d\udd34\ud83d\udd34\u26aa\u26aa\u26aa
> Importance: \ud83d\udd35\ud83d\udd35\ud83d\udd35\u26aa\u26aa
>
> You shouldn't spend more than 10-25 minutes on these exercises.
> ```

Fill in the functions below to detect current-token, previous-token, and first-token heads. Use the average attention probability along the relevant diagonal/column. Validate your detectors by comparing to the visual patterns above.

<details>
<summary>Hint</summary>

Use `torch.diagonal` with appropriate `offset` parameter. For previous-token heads, use offset `-1`. For current-token heads, use the main diagonal (offset `0`). For first-token heads, look at column 0.
</details>

<details>
<summary>Solution</summary>

```python
def current_attn_detector(patterns: dict[int, Tensor]) -> list[str]:
    attn_heads = []
    for layer in patterns:
        pattern = patterns[layer].squeeze(0)
        for head in range(pattern.shape[0]):
            score = pattern[head].diagonal().mean()
            if score > 0.4:
                attn_heads.append(f"{layer}.{head}")
    return attn_heads

def prev_attn_detector(patterns: dict[int, Tensor]) -> list[str]:
    attn_heads = []
    for layer in patterns:
        pattern = patterns[layer].squeeze(0)
        for head in range(pattern.shape[0]):
            score = pattern[head].diagonal(-1).mean()
            if score > 0.4:
                attn_heads.append(f"{layer}.{head}")
    return attn_heads

def first_attn_detector(patterns: dict[int, Tensor]) -> list[str]:
    attn_heads = []
    for layer in patterns:
        pattern = patterns[layer].squeeze(0)
        for head in range(pattern.shape[0]):
            score = pattern[head][:, 0].mean()
            if score > 0.4:
                attn_heads.append(f"{layer}.{head}")
    return attn_heads
```
</details>

In [18]:
# Cell 18: Exercise - attention pattern detectors
def current_attn_detector(patterns: dict[int, Tensor]) -> list[str]:
    """Detects current-token heads (strong diagonal pattern)."""
    pass


def prev_attn_detector(patterns: dict[int, Tensor]) -> list[str]:
    """Detects previous-token heads (strong sub-diagonal pattern)."""
    pass


def first_attn_detector(patterns: dict[int, Tensor]) -> list[str]:
    """Detects first-token heads (strong attention to position 0)."""
    pass


print("Heads attending to current token  =", ", ".join(current_attn_detector(patterns)))
print("Heads attending to previous token =", ", ".join(prev_attn_detector(patterns)))
print("Heads attending to first token    =", ", ".join(first_attn_detector(patterns)))

TypeError: can only join an iterable

## What are induction heads?

(Note: I use induction **head** to refer to the head in the second layer which attends to the 'token immediately after the copy of the current token', and induction **circuit** to refer to the circuit consisting of the composition of a **previous token head** in layer 0 and an **induction head** in layer 1)

[Induction heads](https://transformer-circuits.pub/2021/framework/index.html#induction-heads) are the first sophisticated circuit we see in transformers! The induction circuit consists of a previous token head in layer 0 and an induction head in layer 1, where the induction head learns to attend to the token immediately *after* copies of the current token via K-Composition with the previous token head.

<details>
<summary>An aside on why induction heads are a big deal</summary>

- They develop fairly suddenly in a **phase change** - a striking divergence from 1L model behaviour.
- They are responsible for a **significant loss decrease** - visible as a bump in the loss curve.
- They seem responsible for the vast majority of **in-context learning**.
- The same core circuit appears in more sophisticated settings like translation or few-shot learning.
</details>

##### Question - why couldn't an induction head form in a 1L model?

<details>
<summary>Answer</summary>

Because this would require a head which attends to a key position based on the *value of the token before it*. Attention scores are just a function of the key token and the query token, and are not a function of other tokens.
</details>

## Checking for the induction capability

A striking thing about models with induction heads is that, given a repeated sequence of random tokens, they can predict the repeated half of the sequence. This is nothing like its training data, so this is kind of wild!

### Exercise - plot per-token loss on repeated sequence

> ```yaml
> Difficulty: \ud83d\udd34\ud83d\udd34\u26aa\u26aa\u26aa
> Importance: \ud83d\udd35\ud83d\udd35\u26aa\u26aa\u26aa
>
> You shouldn't spend more than 10-15 minutes on these exercises.
> ```

Fill in `generate_repeated_tokens` to create sequences of the form `[BOS, rand_tokens, rand_tokens]`, then run the model and compare per-token loss on the two halves.

<details>
<summary>Hint</summary>

Use `torch.randint(0, cfg.d_vocab, (batch_size, seq_len))` for the random half. Concatenate `[prefix, half, half]` using `torch.cat`.
</details>

<details>
<summary>Solution</summary>

```python
def generate_repeated_tokens(seq_len: int, batch_size: int = 1) -> Int[Tensor, "batch full_seq"]:
    torch.manual_seed(0)
    prefix = (torch.ones(batch_size, 1) * tokenizer.bos_token_id).long()
    half = torch.randint(0, cfg.d_vocab, (batch_size, seq_len), dtype=torch.int64)
    return torch.cat([prefix, half, half], dim=-1).to(device)

def get_log_probs(logits, tokens):
    logprobs = logits.log_softmax(dim=-1)
    return eindex(logprobs, tokens, "b s [b s+1]")
```
</details>

In [ ]:
# Cell 19: Exercise - generate repeated tokens and log prob helpers
def generate_repeated_tokens(seq_len: int, batch_size: int = 1) -> Int[Tensor, "batch full_seq"]:
    """Generates [BOS, rand_tokens, rand_tokens] sequences."""
    pass


def get_log_probs(
    logits: Float[Tensor, "batch posn d_vocab"],
    tokens: Int[Tensor, "batch posn"],
) -> Float[Tensor, "batch posn-1"]:
    pass

In [ ]:
# Cell 20: Run on repeated tokens, compare first vs second half
seq_len = 50
rep_tokens = generate_repeated_tokens(seq_len, batch_size=1)

# Initialize variables BEFORE trace
rep_patterns = {}
rep_embed = None
rep_pos_embed = None
rep_logits = None

with model.trace(rep_tokens):
    rep_embed = model.embed.output.save()
    rep_pos_embed = model.pos_embed.output.save()
    for layer in range(cfg.n_layers):
        rep_patterns[layer] = model.blocks[layer].attn.hook_pattern.output.save()
    rep_logits = model.output.save()

log_probs = get_log_probs(rep_logits, rep_tokens).squeeze()

print(f"Performance on the first half: {log_probs[:seq_len].mean():.3f}")
print(f"Performance on the second half: {log_probs[seq_len:].mean():.3f}")

In [ ]:
# Cell 21: Plot per-token loss
rep_str = [tokenizer.decode(t) for t in rep_tokens[0].tolist()]

fig = px.line(y=to_numpy(-log_probs), title="Per-token log prob on repeated sequence")
fig.add_vrect(x0=seq_len-0.5, x1=2*seq_len-0.5, fillcolor="green", opacity=0.1,
              annotation_text="Second half (should be predicted)")
fig.show()

### Looking for Induction Attention Patterns

The characteristic pattern of induction heads is a diagonal stripe, with the diagonal offset as `seq_len-1` (because the destination token attends to the token *after* the destination token's previous occurrence).

You should see that heads 4 and 10 are strongly induction-y in layer 1.

In [ ]:
# Cell 22: Visualize attention on repeated tokens
for layer in range(cfg.n_layers):
    attention_pattern = rep_patterns[layer].squeeze(0)
    display(cv.attention.attention_heads(
        tokens=rep_str, attention=attention_pattern.cpu(),
        attention_head_names=[f"L{layer}H{i}" for i in range(cfg.n_heads)],
    ))

### Exercise - make an induction-head detector

> ```yaml
> Difficulty: \ud83d\udd34\ud83d\udd34\u26aa\u26aa\u26aa
> Importance: \ud83d\udd35\ud83d\udd35\u26aa\u26aa\u26aa
>
> You shouldn't spend more than 5-15 minutes on this exercise.
> ```

Make an induction pattern score function, which looks for the average attention paid to the offset diagonal.

<details>
<summary>Help - I'm not sure what offset to use.</summary>

The offset should be `-(seq_len-1)`, because the second instance of random token `T` will attend to the token **after** the first instance of `T`.
</details>

<details>
<summary>Solution</summary>

```python
def induction_attn_detector(patterns: dict[int, Tensor], seq_len: int) -> list[str]:
    attn_heads = []
    for layer in patterns:
        pattern = patterns[layer].squeeze(0)
        for head in range(pattern.shape[0]):
            score = pattern[head].diagonal(-seq_len + 1).mean()
            if score > 0.4:
                attn_heads.append(f"{layer}.{head}")
    return attn_heads
```
</details>

In [ ]:
# Cell 23: Exercise - induction head detector
def induction_attn_detector(patterns: dict[int, Tensor], seq_len: int) -> list[str]:
    """Detects induction heads by looking for the characteristic offset-diagonal stripe."""
    pass


print("Induction heads =", ", ".join(induction_attn_detector(rep_patterns, seq_len)))

---
# Section 3: nnsight Interventions

> ##### Learning Objectives
>
> - Use nnsight's trace context to access and modify activations
> - Build tools to perform attribution, i.e. detecting which components are responsible for performance on a given task
> - Understand how to perform basic interventions like **ablation** using nnsight

## nnsight Trace Context

In TransformerLens, you use `model.run_with_hooks(tokens, fwd_hooks=[(name, hook_fn)])` to access and modify activations. In nnsight, the trace context serves the same purpose:

```python
# TransformerLens style:
def hook_fn(pattern, hook):
    induction_score_store[hook.layer()] = pattern.diagonal(...).mean()
model.run_with_hooks(tokens, fwd_hooks=[(filter, hook_fn)])

# nnsight style:
with model.trace(tokens):
    pattern = model.blocks[layer].attn.hook_pattern.output.save()
# Then compute induction scores from the saved pattern
```

The key advantage of nnsight: you directly read/write module outputs inside the trace, rather than defining separate hook functions.

### Exercise - calculate induction scores with nnsight

> ```yaml
> Difficulty: \ud83d\udd34\ud83d\udd34\ud83d\udd34\u26aa\u26aa
> Importance: \ud83d\udd35\ud83d\udd35\ud83d\udd35\ud83d\udd35\u26aa
>
> You shouldn't spend more than 15-20 minutes on this exercise.
> ```

Save all attention patterns in a single trace, then compute the induction score for each head. The induction score is the average attention paid along the `(1 - seq_len)` offset diagonal.

<details>
<summary>Hint</summary>

Use `pattern.diagonal(dim1=-2, dim2=-1, offset=1 - seq_len)` to get the induction stripe, then reduce with `einops.reduce(stripe, "batch head pos -> head", "mean")`.
</details>

<details>
<summary>Solution</summary>

```python
saved_patterns = {}
with model.trace(rep_tokens_10):
    for layer in range(cfg.n_layers):
        saved_patterns[layer] = model.blocks[layer].attn.hook_pattern.output.save()

induction_score_store = torch.zeros(cfg.n_layers, cfg.n_heads, device=device)
for layer in range(cfg.n_layers):
    pattern = saved_patterns[layer]
    induction_stripe = pattern.diagonal(dim1=-2, dim2=-1, offset=1 - seq_len)
    induction_score = einops.reduce(induction_stripe, "batch head pos -> head", "mean")
    induction_score_store[layer] = induction_score
```
</details>

In [ ]:
# Cell 24: Exercise - induction scores via nnsight trace
seq_len = 50
batch_size = 10
rep_tokens_10 = generate_repeated_tokens(seq_len, batch_size)

# Initialize variables BEFORE trace
saved_patterns = {}

with model.trace(rep_tokens_10):
    for layer in range(cfg.n_layers):
        saved_patterns[layer] = model.blocks[layer].attn.hook_pattern.output.save()

# Compute induction scores after the trace
induction_score_store = torch.zeros(cfg.n_layers, cfg.n_heads, device=device)

for layer in range(cfg.n_layers):
    pattern = saved_patterns[layer]
    induction_stripe = pattern.diagonal(dim1=-2, dim2=-1, offset=1 - seq_len)
    induction_score = einops.reduce(induction_stripe, "batch head pos -> head", "mean")
    induction_score_store[layer] = induction_score

imshow(
    induction_score_store,
    labels={"x": "Head", "y": "Layer"},
    title="Induction Score by Head",
    text_auto=".2f", width=900, height=350,
)

### Exercise - find induction heads in GPT2-small

> ```yaml
> Difficulty: \ud83d\udd34\ud83d\udd34\ud83d\udd34\u26aa\u26aa
> Importance: \ud83d\udd35\ud83d\udd35\ud83d\udd35\ud83d\udd35\u26aa
>
> You shouldn't spend more than 10-20 minutes on this exercise.
> ```

Perform the same analysis on GPT-2 Small. You should observe that some heads in the middle layers have high induction scores. For GPT-2, we compute attention patterns from the fused Q/K/V projection (`c_attn`).

<details>
<summary>Solution</summary>

```python
# See code cells below - save c_attn outputs for all 12 layers, compute patterns,
# then extract the induction diagonal.
```
</details>

In [ ]:
# Cell 25: Helper to compute GPT-2 attention patterns from fused QKV
def compute_gpt2_attention_patterns(qkv: Tensor) -> Tensor:
    """Compute attention patterns from GPT-2's fused c_attn output."""
    d_model, n_heads, d_head = 768, 12, 64
    q, k, _ = qkv.split(d_model, dim=-1)
    batch, seq = q.shape[:2]
    q = q.view(batch, seq, n_heads, d_head).permute(0, 2, 1, 3)
    k = k.view(batch, seq, n_heads, d_head).permute(0, 2, 1, 3)
    scores = torch.einsum("bhqd,bhkd->bhqk", q, k) / (d_head ** 0.5)
    causal_mask = torch.triu(torch.ones(seq, seq, dtype=torch.bool, device=scores.device), diagonal=1)
    scores.masked_fill_(causal_mask, -1e9)
    return F.softmax(scores, dim=-1)

In [ ]:
# Cell 26: Exercise - find induction heads in GPT-2 Small
seq_len = 50
batch_size = 10
rep_tokens_gpt2 = generate_repeated_tokens(seq_len, batch_size)

qkv_all_layers = {}

with gpt2.trace(rep_tokens_gpt2):
    for layer in range(12):
        qkv_all_layers[layer] = gpt2.transformer.h[layer].attn.c_attn.output.save()

induction_score_store_gpt2 = torch.zeros(12, 12, device=device)

for layer in range(12):
    pattern = compute_gpt2_attention_patterns(qkv_all_layers[layer])
    induction_stripe = pattern.diagonal(dim1=-2, dim2=-1, offset=1 - seq_len)
    induction_score = einops.reduce(induction_stripe, "batch head pos -> head", "mean")
    induction_score_store_gpt2[layer] = induction_score

imshow(
    induction_score_store_gpt2,
    labels={"x": "Head", "y": "Layer"},
    title="Induction Score by Head (GPT-2 Small)",
    text_auto=".1f", width=700, height=500,
)

In [ ]:
# Cell 27: Visualize GPT-2 induction head layers
rep_tokens_single = generate_repeated_tokens(seq_len, batch_size=1)
rep_str_gpt2 = [gpt2.tokenizer.decode(t) for t in rep_tokens_single[0].tolist()]

qkv_viz = {}

with gpt2.trace(rep_tokens_single):
    for layer in [5, 6, 7]:
        qkv_viz[layer] = gpt2.transformer.h[layer].attn.c_attn.output.save()

for layer in [5, 6, 7]:
    pattern = compute_gpt2_attention_patterns(qkv_viz[layer]).squeeze(0)
    print(f"Layer {layer}:")
    display(cv.attention.attention_heads(
        tokens=rep_str_gpt2, attention=pattern.cpu(),
        attention_head_names=[f"L{layer}H{i}" for i in range(12)],
    ))

## Building interpretability tools

### Direct Logit Attribution

A consequence of the residual stream is that the output logits are the sum of the contributions of each layer, and thus the sum of the results of each head. This means we can decompose the output logits into a term coming from each head.

Your components are:
- The direct path (embedding to unembedding via residual connections)
- Each layer 0 head (via the residual connection, skipping layer 1)
- Each layer 1 head

### Exercise - build logit attribution tool

> ```yaml
> Difficulty: \ud83d\udd34\ud83d\udd34\ud83d\udd34\u26aa\u26aa
> Importance: \ud83d\udd35\ud83d\udd35\ud83d\udd35\ud83d\udd35\u26aa
>
> You shouldn't spend more than 10-15 minutes on this exercise.
> ```

Implement `logit_attribution` which returns the contribution of each component in the "correct direction". The function takes embeddings, layer 0 results, layer 1 results, the unembedding matrix, and the token sequence.

<details>
<summary>Solution</summary>

```python
def logit_attribution(embed, l1_results, l2_results, W_U, tokens):
    W_U_correct_tokens = W_U[:, tokens[1:]]
    direct = einops.einsum(W_U_correct_tokens, embed[:-1], "emb seq, seq emb -> seq")
    l1_attr = einops.einsum(W_U_correct_tokens, l1_results[:-1], "emb seq, seq nhead emb -> seq nhead")
    l2_attr = einops.einsum(W_U_correct_tokens, l2_results[:-1], "emb seq, seq nhead emb -> seq nhead")
    return torch.cat([direct.unsqueeze(-1), l1_attr, l2_attr], dim=-1)
```
</details>

In [ ]:
# Cell 28: Exercise - logit attribution function
def logit_attribution(
    embed: Float[Tensor, "seq d_model"],
    l1_results: Float[Tensor, "seq nheads d_model"],
    l2_results: Float[Tensor, "seq nheads d_model"],
    W_U: Float[Tensor, "d_model d_vocab"],
    tokens: Int[Tensor, "seq"],
) -> Float[Tensor, "seq-1 n_components"]:
    """
    Returns logit attributions of shape (seq-1, 1 + 2*n_heads).
    Columns: [direct_path, L0H0, ..., L0H11, L1H0, ..., L1H11]
    """
    pass

In [ ]:
# Cell 29: Compute and verify logit attribution
text = "We think that powerful, significantly superhuman machine intelligence is more likely than not to be created this century. If current machine learning techniques were scaled up to this level, we think they would by default produce systems that are deceptive or manipulative, and that no solid plans are known for how to avoid this."
tokens = tokenizer(text, return_tensors="pt")["input_ids"].to(device)

embed = None
l1_results = None
l2_results = None
logits_saved = None

with model.trace(tokens):
    embed = model.embed.output.save()
    l1_results = model.blocks[0].attn.hook_result.output.save()
    l2_results = model.blocks[1].attn.hook_result.output.save()
    logits_saved = model.output.save()

embed_sq = embed.squeeze(0)
l1_results_sq = l1_results.squeeze(0)
l2_results_sq = l2_results.squeeze(0)

W_U = raw_model.unembed.W_U

logit_attr = logit_attribution(embed_sq, l1_results_sq, l2_results_sq, W_U, tokens.squeeze())

correct_token_logits = logits_saved[0, torch.arange(len(tokens[0]) - 1), tokens[0, 1:]]
torch.testing.assert_close(logit_attr.sum(1), correct_token_logits, atol=1e-2, rtol=0)
print("Logit attribution test passed!")

In [ ]:
# Cell 30: Plot logit attribution (demo prompt)
str_tokens = [tokenizer.decode(t) for t in tokens[0].tolist()]
component_labels = ["Direct"] + [f"L0H{i}" for i in range(cfg.n_heads)] + [f"L1H{i}" for i in range(cfg.n_heads)]

imshow(
    logit_attr.T,
    x=[str_tokens[i] for i in range(1, len(str_tokens))],
    y=component_labels,
    labels={"x": "Token", "y": "Component", "color": "Logit contribution"},
    title="Logit attribution (demo prompt)",
    width=1200, height=500,
)

### Exercise - interpret logit attribution for the induction heads

> ```yaml
> Difficulty: \ud83d\udd34\ud83d\udd34\ud83d\udd34\u26aa\u26aa
> Importance: \ud83d\udd35\ud83d\udd35\ud83d\udd35\ud83d\udd35\u26aa
>
> You shouldn't spend more than 10-15 minutes on this exercise.
> ```

Perform logit attribution on the repeated token sequence. What do you expect to see?

<details>
<summary>Interpretation</summary>

In the second half, heads `1.4` and `1.10` have large logit attribution scores - they are performing induction (using the information from the first occurrence to predict the second). The first half is mostly meaningless since the sequences are random.
</details>

In [ ]:
# Cell 31: Exercise - logit attribution on induction prompt
seq_len = 50
rep_tokens = generate_repeated_tokens(seq_len, batch_size=1)

rep_embed = None
rep_l1_results = None
rep_l2_results = None
rep_logits = None

with model.trace(rep_tokens):
    rep_embed = model.embed.output.save()
    rep_l1_results = model.blocks[0].attn.hook_result.output.save()
    rep_l2_results = model.blocks[1].attn.hook_result.output.save()
    rep_logits = model.output.save()

logit_attr_rep = logit_attribution(
    rep_embed.squeeze(0), rep_l1_results.squeeze(0), rep_l2_results.squeeze(0),
    W_U, rep_tokens.squeeze(),
)

rep_str = [tokenizer.decode(t) for t in rep_tokens[0].tolist()]
imshow(
    logit_attr_rep.T,
    x=[rep_str[i] for i in range(1, len(rep_str))],
    y=component_labels,
    labels={"x": "Token", "y": "Component", "color": "Logit contribution"},
    title="Logit attribution (random induction prompt)",
    width=1200, height=500,
)

## Ablations

An ablation is a simple causal intervention - we pick some part of the model and set it to zero (or to its mean). This is a crude proxy for how much that part matters. In nnsight, we directly modify activations inside the trace context.

### Exercise - induction head ablation

> ```yaml
> Difficulty: \ud83d\udd34\ud83d\udd34\u26aa\u26aa\u26aa
> Importance: \ud83d\udd35\ud83d\udd35\ud83d\udd35\ud83d\udd35\u26aa
>
> You should aim to spend 20-35 mins on this exercise.
> ```

Implement `get_ablation_scores` which returns the increase in cross-entropy loss from ablating each head. In nnsight, you modify the `hook_z` output directly inside the trace:

```python
with model.trace(tokens):
    model.blocks[layer].attn.hook_z.output[:, :, head, :] = 0.0  # zero ablation
    ablated_logits = model.output.save()
```

<details>
<summary>Interpretation of results</summary>

Head `0.7` is by far the most important in layer 0 (strongest previous token head). Heads `1.4` and `1.10` are most important in layer 1 (the induction heads). This is evidence from **causal intervention**, which is stronger than the observational evidence from attention patterns alone.
</details>

<details>
<summary>Solution</summary>

```python
def get_ablation_scores(model, tokens, ablation_type="zero"):
    seq_len = (tokens.shape[1] - 1) // 2
    ablation_scores = torch.zeros(cfg.n_layers, cfg.n_heads, device=device)

    with model.trace(tokens):
        clean_logits = model.output.save()
    loss_no_ablation = -get_log_probs(clean_logits, tokens)[:, -(seq_len - 1):].mean()

    for layer in tqdm(range(cfg.n_layers)):
        for head in range(cfg.n_heads):
            with model.trace(tokens):
                if ablation_type == "zero":
                    model.blocks[layer].attn.hook_z.output[:, :, head, :] = 0.0
                elif ablation_type == "mean":
                    z_head = model.blocks[layer].attn.hook_z.output[:, :, head, :]
                    mean_z = z_head.mean(dim=1, keepdim=True)
                    model.blocks[layer].attn.hook_z.output[:, :, head, :] = mean_z
                ablated_logits = model.output.save()
            loss = -get_log_probs(ablated_logits, tokens)[:, -(seq_len - 1):].mean()
            ablation_scores[layer, head] = loss - loss_no_ablation
    return ablation_scores
```
</details>

In [ ]:
# Cell 32: Exercise - ablation scoring function
def get_ablation_scores(
    model: NNsight,
    tokens: Int[Tensor, "batch seq"],
    ablation_type: str = "zero",
) -> Float[Tensor, "n_layers n_heads"]:
    """
    Returns increase in cross-entropy loss from ablating each head.
    ablation_type: 'zero' (set to 0) or 'mean' (replace with mean across positions).
    """
    pass

In [ ]:
# Cell 33: Zero ablation scores
seq_len = 50
rep_tokens = generate_repeated_tokens(seq_len, batch_size=1)

ablation_scores = get_ablation_scores(model, rep_tokens, ablation_type="zero")

imshow(
    ablation_scores,
    labels={"x": "Head", "y": "Layer", "color": "Loss diff"},
    title="Loss Difference After Zero-Ablating Heads",
    text_auto=".2f", width=900, height=350,
)

### Exercise - mean ablation

> ```yaml
> Difficulty: \ud83d\udd34\u26aa\u26aa\u26aa\u26aa
> Importance: \ud83d\udd35\ud83d\udd35\ud83d\udd35\u26aa\u26aa
>
> You should aim to spend 5-15 mins on this exercise.
> ```

An alternative to zero-ablation is **mean-ablation**, where we set values to their mean across positions. This can be more informative because zero-ablation takes the model out of its normal distribution. You should see slightly cleaner results.

<details>
<summary>Solution</summary>

Just call `get_ablation_scores` with `ablation_type="mean"` and a larger batch size.
</details>

In [ ]:
# Cell 34: Exercise - mean ablation scores
rep_tokens_batch = generate_repeated_tokens(seq_len=50, batch_size=10)

mean_ablation_scores = get_ablation_scores(model, rep_tokens_batch, ablation_type="mean")

imshow(
    mean_ablation_scores,
    labels={"x": "Head", "y": "Layer", "color": "Loss diff"},
    title="Loss Difference After Mean-Ablating Heads",
    text_auto=".2f", width=900, height=350,
)

---
# Section 4: Reverse-engineering induction circuits

> ##### Learning Objectives
>
> - Understand the difference between investigating a circuit by looking at activation patterns, and reverse-engineering a circuit by looking directly at the weights
> - Use the FactoredMatrix class to inspect the QK and OV circuits within an induction circuit
> - Perform further exploration of induction circuits: composition scores, and targeted ablations

## Refresher - the induction circuit

We've found:
- Heads `1.4` and `1.10` have the characteristic induction head attention pattern
- Head `0.7` strongly attends to the previous token
- Logit attribution shows `1.4` and `1.10` are important for predictions in the second half
- Ablation of `0.7`, `1.4`, and `1.10` all degrade performance

The algorithm:
1. Head `0.7` is a **previous token head** - its QK-circuit always attends to the previous token
2. The **OV circuit** of `0.7` writes a copy of the previous token in a different subspace
3. This output is used by the **key** input of `1.10` (via **K-Composition**) to attend to 'the token whose previous token matches the current token'
4. The **OV circuit** of `1.10` copies the source token's value to the output logits

## [1] OV copying circuit

The OV circuit determines **what a head writes** when it attends to a token.

Full OV circuit: $W_E W_{OV}^{1.4} W_U$ maps input tokens to output logits. We expect this to be approximately diagonal (i.e., when head `1.4` attends to token `B`, it writes a vector that predicts `B`).

### Exercise - compute OV circuit for `1.4`

> ```yaml
> Difficulty: \ud83d\udd34\ud83d\udd34\u26aa\u26aa\u26aa
> Importance: \ud83d\udd35\ud83d\udd35\ud83d\udd35\u26aa\u26aa
>
> You should spend up to ~10 minutes on this exercise.
> ```

Compute the full OV circuit as a `FactoredMatrix` object: `W_E @ FactoredMatrix(W_V, W_O) @ W_U`.

<details>
<summary>Solution</summary>

```python
W_V = raw_model.blocks[1].attn.W_V[4]
W_O = raw_model.blocks[1].attn.W_O[4]
OV_circuit = FactoredMatrix(W_V, W_O)
full_OV_circuit = raw_model.embed.W_E @ OV_circuit @ raw_model.unembed.W_U
```
</details>

In [ ]:
# Cell 35: Exercise - OV circuit for copying head
head_index = 4
layer = 1

W_V = raw_model.blocks[layer].attn.W_V[head_index]
W_O = raw_model.blocks[layer].attn.W_O[head_index]
W_E = raw_model.embed.W_E
W_U = raw_model.unembed.W_U

OV_circuit = FactoredMatrix(W_V, W_O)
full_OV_circuit = W_E @ OV_circuit @ W_U

print(f"Full OV circuit shape: {full_OV_circuit.shape}")

In [ ]:
# Cell 36: Visualize OV circuit diagonal
indices = torch.randint(0, cfg.d_vocab, (200,))
full_OV_circuit_sample = full_OV_circuit[indices, indices].AB

imshow(
    full_OV_circuit_sample,
    labels={"x": "Logits on output token", "y": "Input token"},
    title="Full OV circuit for copying head (L1H4)",
    width=700, height=600,
)

### Exercise - compute circuit accuracy

> ```yaml
> Difficulty: \ud83d\udd34\ud83d\udd34\u26aa\u26aa\u26aa
> Importance: \ud83d\udd35\ud83d\udd35\u26aa\u26aa\u26aa
>
> You should spend approximately 10-15 minutes on this exercise.
> ```

Implement `top_1_acc` - what fraction of the time is the largest logit on the diagonal?

<details>
<summary>Solution</summary>

```python
def top_1_acc(full_OV_circuit: FactoredMatrix, batch_size: int = 1000) -> float:
    total = 0
    for indices in torch.split(torch.arange(full_OV_circuit.shape[0], device=device), batch_size):
        AB_slice = full_OV_circuit[indices].AB
        total += (torch.argmax(AB_slice, dim=1) == indices).float().sum().item()
    return total / full_OV_circuit.shape[0]
```
</details>

In [ ]:
# Cell 37: Exercise - top-1 accuracy of OV circuit
def top_1_acc(full_OV_circuit: FactoredMatrix, batch_size: int = 1000) -> float:
    """Fraction of tokens where the argmax logit equals the input token (diagonal)."""
    pass


print(f"Fraction of time best logit is on diagonal (L1H4): {top_1_acc(full_OV_circuit):.4f}")

### Exercise - compute effective circuit

> ```yaml
> Difficulty: \ud83d\udd34\ud83d\udd34\u26aa\u26aa\u26aa
> Importance: \ud83d\udd35\ud83d\udd35\ud83d\udd35\u26aa\u26aa
>
> You shouldn't spend more than 5-10 minutes on this exercise.
> ```

Since both `1.4` and `1.10` have the same attention pattern, the effective OV circuit is $W_E(W_V^{1.4}W_O^{1.4}+W_V^{1.10}W_O^{1.10})W_U$. This should have much higher accuracy!

<details>
<summary>Question - why might the model split the circuit across two heads?</summary>

Because $W_V W_O$ is a rank 64 matrix. The sum of two is rank 128 - a significantly better approximation to the desired 50K x 50K identity-like matrix!
</details>

In [ ]:
# Cell 38: Exercise - combined OV circuit for both induction heads
W_O_both = einops.rearrange(
    raw_model.blocks[1].attn.W_O[[4, 10]], "head d_head d_model -> (head d_head) d_model"
)
W_V_both = einops.rearrange(
    raw_model.blocks[1].attn.W_V[[4, 10]], "head d_model d_head -> d_model (head d_head)"
)

W_E = raw_model.embed.W_E
W_U = raw_model.unembed.W_U
W_OV_eff = W_E @ FactoredMatrix(W_V_both, W_O_both) @ W_U

print(f"Combined OV accuracy: {top_1_acc(W_OV_eff):.4f}")

## [2] QK prev-token circuit

The QK circuit determines **what a head attends to**. For the positional QK circuit:

$$W_{\text{pos}} W_Q^{0.7} (W_K^{0.7})^T W_{\text{pos}}^T$$

This should show a strong sub-diagonal pattern (each position attends to the previous position).

### Exercise - interpret full QK-circuit for `0.7`

> ```yaml
> Difficulty: \ud83d\udd34\ud83d\udd34\ud83d\udd34\u26aa\u26aa
> Importance: \ud83d\udd35\ud83d\udd35\ud83d\udd35\u26aa\u26aa
>
> You shouldn't spend more than 10-15 minutes on this exercise.
> ```

In [ ]:
# Cell 39: Exercise - QK circuit for previous-token head
layer = 0
head_index = 7

W_pos = raw_model.pos_embed.W_pos
W_Q_head = raw_model.blocks[layer].attn.W_Q[head_index]
W_K_head = raw_model.blocks[layer].attn.W_K[head_index]

W_QK = W_Q_head @ W_K_head.T
pos_by_pos_scores = W_pos @ W_QK @ W_pos.T

mask = torch.tril(torch.ones_like(pos_by_pos_scores)).bool()
pos_by_pos_pattern = torch.where(
    mask, pos_by_pos_scores / cfg.d_head ** 0.5, torch.tensor(-1e6, device=device)
).softmax(-1)

print(f"Avg lower-diagonal value: {pos_by_pos_pattern.diag(-1).mean():.4f}")

imshow(
    pos_by_pos_pattern[:200, :200],
    labels={"x": "Key", "y": "Query"},
    title="Attention patterns for prev-token QK circuit (L0H7), first 200 positions",
    width=700, height=600,
)

## [3] K-composition circuit

Now the hard part - demonstrating K-Composition between the previous token head and the induction head.

The QK-input for layer 1 is the sum of 14 terms: token embedding + positional embedding + 12 layer-0 head outputs. We decompose this to find which components drive the induction pattern.

### Exercise - analyse the relative importance

> ```yaml
> Difficulty: \ud83d\udd34\ud83d\udd34\ud83d\udd34\ud83d\udd34\u26aa
> Importance: \ud83d\udd35\ud83d\udd35\ud83d\udd35\ud83d\udd35\u26aa
>
> You shouldn't spend more than 15-25 minutes on this exercise.
> ```

Implement `decompose_qk_input`, `decompose_q`, and `decompose_k` to split the QK input into its 14 components.

<details>
<summary>Solution</summary>

```python
def decompose_qk_input(embed, pos_embed, l0_result):
    embed_sq = embed.squeeze(0)
    l0_per_head = l0_result.squeeze(0).permute(1, 0, 2)
    return torch.cat([embed_sq.unsqueeze(0), pos_embed.unsqueeze(0), l0_per_head], dim=0)

def decompose_q(decomposed_input, ind_head_index):
    W_Q = raw_model.blocks[1].attn.W_Q[ind_head_index]
    return einops.einsum(decomposed_input, W_Q, "n seq d_model, d_model d_head -> n seq d_head")

def decompose_k(decomposed_input, ind_head_index):
    W_K = raw_model.blocks[1].attn.W_K[ind_head_index]
    return einops.einsum(decomposed_input, W_K, "n seq d_model, d_model d_head -> n seq d_head")
```
</details>

In [ ]:
# Cell 40: Save activations for QK decomposition
seq_len = 50
rep_tokens = generate_repeated_tokens(seq_len, batch_size=1)

embed_saved = None
pos_embed_saved = None
l0_result = None
q_saved = None
k_saved = None

with model.trace(rep_tokens):
    embed_saved = model.embed.output.save()
    pos_embed_saved = model.pos_embed.output.save()
    l0_result = model.blocks[0].attn.hook_result.output.save()
    q_saved = model.blocks[1].attn.hook_q.output.save()
    k_saved = model.blocks[1].attn.hook_k.output.save()

In [ ]:
# Cell 41: Exercise - QK decomposition functions
def decompose_qk_input(
    embed: Tensor, pos_embed: Tensor, l0_result: Tensor,
) -> Float[Tensor, "n_components seq d_model"]:
    """Decompose the QK input into: [embed, pos_embed, head0_result, ..., head11_result]"""
    pass


def decompose_q(
    decomposed_input: Float[Tensor, "comp seq d_model"],
    ind_head_index: int,
) -> Float[Tensor, "comp seq d_head"]:
    pass


def decompose_k(
    decomposed_input: Float[Tensor, "comp seq d_model"],
    ind_head_index: int,
) -> Float[Tensor, "comp seq d_head"]:
    pass

In [ ]:
# Cell 42: Verify QK decomposition
ind_head_index = 4

decomposed_qk_input = decompose_qk_input(embed_saved, pos_embed_saved, l0_result)
decomposed_q_vals = decompose_q(decomposed_qk_input, ind_head_index)
decomposed_k_vals = decompose_k(decomposed_qk_input, ind_head_index)

torch.testing.assert_close(
    decomposed_q_vals.sum(0), q_saved.squeeze(0)[:, ind_head_index],
    rtol=0.01, atol=0.001,
)
torch.testing.assert_close(
    decomposed_k_vals.sum(0), k_saved.squeeze(0)[:, ind_head_index],
    rtol=0.01, atol=0.01,
)
print("Decomposition tests passed!")

In [ ]:
# Cell 43: Plot QK component norms
component_labels = ["Embed", "PosEmbed"] + [f"0.{h}" for h in range(cfg.n_heads)]

for decomposed_input, name in [(decomposed_q_vals, "query"), (decomposed_k_vals, "key")]:
    imshow(
        decomposed_input.pow(2).sum(-1),
        labels={"x": "Position", "y": "Component"},
        title=f"Norms of components of {name}",
        y=component_labels,
        width=800, height=400,
    )

### Exercise - decompose attention scores

> ```yaml
> Difficulty: \ud83d\udd34\ud83d\udd34\u26aa\u26aa\u26aa
> Importance: \ud83d\udd35\ud83d\udd35\ud83d\udd35\ud83d\udd35\u26aa
>
> You shouldn't spend more than 5-10 minutes on this exercise.
> ```

<details>
<summary>Solution</summary>

```python
def decompose_attn_scores(decomposed_q, decomposed_k):
    return einops.einsum(
        decomposed_q, decomposed_k,
        "qc qp d, kc kp d -> qc kc qp kp",
    ) / (cfg.d_head ** 0.5)
```
</details>

In [ ]:
# Cell 44: Exercise - decompose attention scores
def decompose_attn_scores(
    decomposed_q: Float[Tensor, "q_comp q_pos d_head"],
    decomposed_k: Float[Tensor, "k_comp k_pos d_head"],
) -> Float[Tensor, "q_comp k_comp q_pos k_pos"]:
    pass


decomposed_scores = decompose_attn_scores(decomposed_q_vals, decomposed_k_vals)

In [ ]:
# Cell 45: Visualize dominant (Embed, 0.7) contribution and std devs
q_label, k_label = "Embed", "0.7"
qi, ki = component_labels.index(q_label), component_labels.index(k_label)

imshow(
    torch.tril(decomposed_scores[qi, ki]),
    title=f"Attention score contributions: query={q_label}, key={k_label}",
    width=700,
)

decomposed_stds = einops.reduce(
    decomposed_scores, "qc kc qp kp -> qc kc", torch.std,
)
imshow(
    decomposed_stds,
    labels={"x": "Key Component", "y": "Query Component"},
    title="Std dev of attention score contributions",
    x=component_labels, y=component_labels,
    width=700,
)

### Exercise - compute the K-comp circuit

> ```yaml
> Difficulty: \ud83d\udd34\ud83d\udd34\ud83d\udd34\u26aa\u26aa
> Importance: \ud83d\udd35\ud83d\udd35\ud83d\udd35\ud83d\udd35\u26aa
>
> You shouldn't spend more than 10-20 minutes on this exercise.
> ```

The full K-composition circuit: $(W_E W_Q^{1.4})$ x $(W_E W_V^{0.7} W_O^{0.7} W_K^{1.4})^T$

This matrix should approximate identity - the query says "I'm looking for the token that followed A", and the key says "I am the token that followed A".

<details>
<summary>Solution</summary>

```python
def find_K_comp_full_circuit(prev_token_head_index, ind_head_index):
    W_E = raw_model.embed.W_E
    W_Q = raw_model.blocks[1].attn.W_Q[ind_head_index]
    W_K = raw_model.blocks[1].attn.W_K[ind_head_index]
    W_V = raw_model.blocks[0].attn.W_V[prev_token_head_index]
    W_O = raw_model.blocks[0].attn.W_O[prev_token_head_index]
    Q = W_E @ W_Q
    K = W_E @ W_V @ W_O @ W_K
    return FactoredMatrix(Q, K.T)
```
</details>

In [ ]:
# Cell 46: Exercise - K-composition full circuit
def find_K_comp_full_circuit(
    prev_token_head_index: int, ind_head_index: int,
) -> FactoredMatrix:
    pass


K_comp_circuit = find_K_comp_full_circuit(prev_token_head_index=7, ind_head_index=4)
print(f"K-comp circuit shape: {K_comp_circuit.shape}")
print(f"Token frac where max-activating key = same token: {top_1_acc(K_comp_circuit.T):.4f}")

## Composition Scores

Composition scores measure how strongly two heads compose: $\frac{\|W_A W_B\|_F}{\|W_A\|_F \cdot \|W_B\|_F}$

### Exercise - calculate composition scores

> ```yaml
> Difficulty: \ud83d\udd34\ud83d\udd34\ud83d\udd34\u26aa\u26aa
> Importance: \ud83d\udd35\ud83d\udd35\ud83d\udd35\u26aa\u26aa
>
> You shouldn't spend more than 15-25 minutes on this exercise.
> ```

<details>
<summary>Solution</summary>

```python
def get_comp_score(W_A, W_B):
    W_A_norm = W_A.pow(2).sum().sqrt()
    W_B_norm = W_B.pow(2).sum().sqrt()
    W_AB_norm = (W_A @ W_B).pow(2).sum().sqrt()
    return (W_AB_norm / (W_A_norm * W_B_norm)).item()
```
</details>

In [ ]:
# Cell 47: Exercise - composition score function
def get_comp_score(
    W_A: Float[Tensor, "in_A out_A"],
    W_B: Float[Tensor, "out_A out_B"],
) -> float:
    pass

In [ ]:
# Cell 48: Q, K, V composition scores between all L0-L1 head pairs
def get_W_OV(layer, head):
    return raw_model.blocks[layer].attn.W_V[head] @ raw_model.blocks[layer].attn.W_O[head]

def get_W_QK(layer, head):
    return raw_model.blocks[layer].attn.W_Q[head] @ raw_model.blocks[layer].attn.W_K[head].T

composition_scores = {
    "Q": torch.zeros(cfg.n_heads, cfg.n_heads, device=device),
    "K": torch.zeros(cfg.n_heads, cfg.n_heads, device=device),
    "V": torch.zeros(cfg.n_heads, cfg.n_heads, device=device),
}

for i in tqdm(range(cfg.n_heads)):
    W_OV_0i = get_W_OV(0, i)
    for j in range(cfg.n_heads):
        W_QK_1j = get_W_QK(1, j)
        composition_scores["Q"][i, j] = get_comp_score(W_OV_0i, W_QK_1j)
        composition_scores["K"][i, j] = get_comp_score(W_OV_0i, W_QK_1j.T)
        composition_scores["V"][i, j] = get_comp_score(W_OV_0i, get_W_OV(1, j))

for comp_type in ["Q", "K", "V"]:
    imshow(
        composition_scores[comp_type],
        labels={"x": "L1 Head", "y": "L0 Head", "color": "Score"},
        title=f"{comp_type} Composition Scores",
        text_auto=".2f", width=700, height=600,
    )

### Exercise - setting a baseline

> ```yaml
> Difficulty: \ud83d\udd34\ud83d\udd34\u26aa\u26aa\u26aa
> Importance: \ud83d\udd35\ud83d\udd35\u26aa\u26aa\u26aa
>
> You shouldn't spend more than ~10 minutes on this exercise.
> ```

Generate random composition scores as a baseline, to understand what values are significant.

<details>
<summary>Solution</summary>

```python
def generate_single_random_comp_score():
    W_A_left = torch.empty(cfg.d_model, cfg.d_head)
    W_B_left = torch.empty(cfg.d_model, cfg.d_head)
    W_A_right = torch.empty(cfg.d_model, cfg.d_head)
    W_B_right = torch.empty(cfg.d_model, cfg.d_head)
    for W in [W_A_left, W_B_left, W_A_right, W_B_right]:
        nn.init.kaiming_uniform_(W, a=np.sqrt(5))
    W_A = W_A_left @ W_A_right.T
    W_B = W_B_left @ W_B_right.T
    return get_comp_score(W_A, W_B)
```
</details>

In [ ]:
# Cell 49: Exercise - random baseline for composition scores
def generate_single_random_comp_score() -> float:
    pass


n_samples = 300
comp_scores_baseline = np.array([generate_single_random_comp_score() for _ in tqdm(range(n_samples))])
print(f"Mean: {comp_scores_baseline.mean():.4f}")
print(f"Std: {comp_scores_baseline.std():.4f}")

hist(comp_scores_baseline, title="Random composition scores", nbins=50, width=800,
     labels={"x": "Composition score"})

In [ ]:
# Cell 50: Composition scores with baseline subtracted
baseline = comp_scores_baseline.mean()

for comp_type in ["Q", "K", "V"]:
    fig = imshow(
        composition_scores[comp_type] - baseline,
        labels={"x": "L1 Head", "y": "L0 Head", "color": "Score - baseline"},
        title=f"{comp_type} Composition Scores (baseline-subtracted)",
        text_auto=".2f", width=700, height=600,
    )

### Bonus: Batched Composition Scores with FactoredMatrix

> ```yaml
> Difficulty: \ud83d\udd34\ud83d\udd34\ud83d\udd34\ud83d\udd34\u26aa
> Importance: \ud83d\udd35\u26aa\u26aa\u26aa\u26aa
>
> This exercise is optional.
> ```

<details>
<summary>Solution</summary>

```python
def get_batched_comp_scores(W_As, W_Bs):
    W_As = FactoredMatrix(
        W_As.A.reshape(-1, 1, *W_As.A.shape[-2:]),
        W_As.B.reshape(-1, 1, *W_As.B.shape[-2:]),
    )
    W_Bs = FactoredMatrix(
        W_Bs.A.reshape(1, -1, *W_Bs.A.shape[-2:]),
        W_Bs.B.reshape(1, -1, *W_Bs.B.shape[-2:]),
    )
    W_ABs = W_As @ W_Bs
    return W_ABs.norm() / (W_As.norm() * W_Bs.norm())
```
</details>

In [ ]:
# Cell 51: Exercise - batched composition scores with FactoredMatrix
def get_batched_comp_scores(
    W_As: FactoredMatrix, W_Bs: FactoredMatrix,
) -> Tensor:
    pass

In [ ]:
# Cell 52: Verify batched scores match loop-based scores
W_Q_all = torch.stack([raw_model.blocks[l].attn.W_Q for l in range(cfg.n_layers)])
W_K_all = torch.stack([raw_model.blocks[l].attn.W_K for l in range(cfg.n_layers)])
W_V_all = torch.stack([raw_model.blocks[l].attn.W_V for l in range(cfg.n_layers)])
W_O_all = torch.stack([raw_model.blocks[l].attn.W_O for l in range(cfg.n_layers)])

W_QK = FactoredMatrix(W_Q_all, W_K_all.transpose(-1, -2))
W_OV = FactoredMatrix(W_V_all, W_O_all)

composition_scores_batched = {
    "Q": get_batched_comp_scores(W_OV[0], W_QK[1]),
    "K": get_batched_comp_scores(W_OV[0], W_QK[1].T),
    "V": get_batched_comp_scores(W_OV[0], W_OV[1]),
}

for comp_type in ["Q", "K", "V"]:
    torch.testing.assert_close(
        composition_scores_batched[comp_type],
        composition_scores[comp_type],
        atol=1e-4, rtol=1e-3,
    )
print("Batched composition scores match!")

## Targeted Ablation to Verify the Circuit

The final test: ablate each L0 head's value output and measure the effect on L1H4's induction score. If our circuit analysis is correct, ablating L0H7 should have the largest negative effect.

In [ ]:
# Cell 53: Targeted ablation to verify the circuit
seq_len = 50
rep_tokens = generate_repeated_tokens(seq_len, batch_size=1)


def ablation_induction_score(
    prev_head_index: Optional[int], ind_head_index: int,
) -> float:
    """Ablate a specific L0 head (set its V output to 0) and measure
    the induction score of the specified L1 head."""
    with model.trace(rep_tokens):
        if prev_head_index is not None:
            model.blocks[0].attn.hook_v.output[:, :, prev_head_index, :] = 0.0
        pattern = model.blocks[1].attn.hook_pattern.output.save()

    return pattern[0, ind_head_index].diag(-(seq_len - 1)).mean().item()


baseline_score = ablation_induction_score(None, 4)
print(f"Induction score for no ablations: {baseline_score:.5f}\n")

for i in range(cfg.n_heads):
    new_score = ablation_induction_score(i, 4)
    change = new_score - baseline_score
    print(f"Ablation score change for head {i:02}: {change:+.5f}")

---
## Summary

We've successfully reverse-engineered the induction circuit in this 2-layer attention-only transformer:

1. **L0H7** is the previous-token head (QK circuit attends to position $i-1$)
2. **L1H4** and **L1H10** are induction heads (OV circuit copies attended-to tokens)
3. **K-composition** between L0H7 and L1H4 creates the induction attention pattern
4. **Ablating L0H7** destroys the induction capability (largest negative score change)

**Key nnsight patterns used:**

| TransformerLens | nnsight equivalent |
|---|---|
| `logits, cache = model.run_with_cache(tokens)` | `with model.trace(tokens): val = model.module.output.save()` |
| `cache["pattern", layer]` | `model.blocks[layer].attn.hook_pattern.output.save()` |
| `cache["result", layer]` | `model.blocks[layer].attn.hook_result.output.save()` |
| `model.run_with_hooks(tokens, fwd_hooks=[(name, fn)])` | Modify outputs directly inside `model.trace()` |
| `z[:, :, head, :] = 0.0` (in hook) | Same code, inside `with model.trace():` |
| `model.W_Q[layer, head]` | `raw_model.blocks[layer].attn.W_Q[head]` |